In [1]:
import sys
from pathlib import Path

ROOT = Path(r"C:\Users\Prabu\Downloads\Katabatic").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT set to:", ROOT)

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from utils import discretize_preprocess

from katabatic.models.tabddpm.models import Tabddpm

ROOT set to: C:\Users\Prabu\Downloads\Katabatic


In [ ]:
# Preprocess data
dataset_path = ROOT / "raw_data" / "nursery.csv"         
output_path = ROOT / "discretized_data" / "nursery.csv"  

output_path.parent.mkdir(parents=True, exist_ok=True)

# Preprocess
discretize_preprocess(str(dataset_path), str(output_path))

In [17]:
import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)         
warnings.filterwarnings("ignore", message="Parameters: {")           
warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")
input_csv = str(output_path)
output_dir = str(ROOT / "sample_data" / "nursery")
real_test_dir = output_dir
synthetic_dir = str(ROOT / "synthetic" / "nursery" / "tabddpm")

class TabDDPMFull(Tabddpm):
    def train(self, *args, **kwargs):

        config = kwargs.get("config") or {}
        if not isinstance(config, dict):
            config = {}

        # 🔧 Tweak training here
        config["steps"] = 3000        
        config["lr"] = 1e-4          
        config["batch_size"] = 256    

        kwargs["config"] = config
        return super().train(*args, **kwargs)


pipeline = TrainTestSplitPipeline(
    model=lambda: TabDDPMFull()
)

result = pipeline.run(
    input_csv=input_csv,
    output_dir=output_dir,
    synthetic_dir=synthetic_dir,
    real_test_dir=real_test_dir,
)

print(result)

Loaded data with shape: (12960, 9)
Saved train/test full data
Train size: (10368, 9), Test size: (2592, 9)
Train label distribution:
 8
0    0.333333
1    0.329186
3    0.312018
4    0.025270
2    0.000193
Name: proportion, dtype: float64
Test label distribution:
 8
0    0.333333
1    0.329090
3    0.312114
4    0.025463
Name: proportion, dtype: float64
Saved X/y split
Training shape: (10368, 8) (10368,)
Test shape: (2592, 8) (2592,)
Step 100/3000 | MLoss: 0.0000 | GLoss: 0.5573
Step 200/3000 | MLoss: 0.0000 | GLoss: 0.0674
Step 300/3000 | MLoss: 0.0000 | GLoss: 0.0077
Step 400/3000 | MLoss: 0.0000 | GLoss: 0.0030
Step 500/3000 | MLoss: 0.0000 | GLoss: 0.0020
Step 600/3000 | MLoss: 0.0000 | GLoss: 0.0013
Step 700/3000 | MLoss: 0.0000 | GLoss: 0.0012
Step 800/3000 | MLoss: 0.0000 | GLoss: 0.0009
Step 900/3000 | MLoss: 0.0000 | GLoss: 0.0008
Step 1000/3000 | MLoss: 0.0000 | GLoss: 0.0007
Step 1100/3000 | MLoss: 0.0000 | GLoss: 0.0007
Step 1200/3000 | MLoss: 0.0000 | GLoss: 0.0007
Step 13

In [23]:
import sys
from pathlib import Path
import warnings
from sklearn.exceptions import ConvergenceWarning

# ----------------- ROOT + imports -----------------
ROOT = Path(r"C:\Users\Prabu\Downloads\Katabatic").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT set to:", ROOT)

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from utils import discretize_preprocess
from katabatic.models.tabddpm.models import Tabddpm

# ----------------- Preprocess nursery -----------------
dataset_path = ROOT / "raw_data" / "nursery.csv"
output_path = ROOT / "discretized_data" / "nursery.csv"

output_path.parent.mkdir(parents=True, exist_ok=True)
discretize_preprocess(str(dataset_path), str(output_path))

# ----------------- Warnings -----------------
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", message="Parameters: {")
warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")

# ----------------- Paths -----------------
input_csv = str(output_path)
output_dir = str(ROOT / "sample_data" / "nursery")
real_test_dir = output_dir
synthetic_dir = str(ROOT / "synthetic" / "nursery" / "tabddpm")

# ----------------- Tuned TabDDPM for nursery -----------------
class TabDDPMNursery(Tabddpm):
    def train(self, *args, **kwargs):
        config = kwargs.get("config") or {}
        if not isinstance(config, dict):
            config = {}

        # 🔧 go back to "full strength" model, just change steps
        config["steps"] = 4000             # more training
        # keep the strong defaults from _defaults:
        # num_timesteps=1000, batch_size=256, use_ema=True, d_layers=(256,256,256,256)

        kwargs["config"] = config
        return super().train(*args, **kwargs)

# ----------------- Pipeline -----------------
pipeline = TrainTestSplitPipeline(
    model=lambda: TabDDPMNursery()
)

result = pipeline.run(
    input_csv=input_csv,
    output_dir=output_dir,
    synthetic_dir=synthetic_dir,
    real_test_dir=real_test_dir,
)

print(result)


ROOT set to: C:\Users\Prabu\Downloads\Katabatic
Preprocessing: C:\Users\Prabu\Downloads\Katabatic\raw_data\nursery.csv
Saved preprocessed discrete dataset to: C:\Users\Prabu\Downloads\Katabatic\discretized_data\nursery.csv
Loaded data with shape: (12960, 9)
Saved train/test full data
Train size: (10368, 9), Test size: (2592, 9)
Train label distribution:
 8
0    0.333333
1    0.329186
3    0.312018
4    0.025270
2    0.000193
Name: proportion, dtype: float64
Test label distribution:
 8
0    0.333333
1    0.329090
3    0.312114
4    0.025463
Name: proportion, dtype: float64
Saved X/y split
Training shape: (10368, 8) (10368,)
Test shape: (2592, 8) (2592,)
Step 100/4000 | MLoss: 0.0000 | GLoss: 0.0123
Step 200/4000 | MLoss: 0.0000 | GLoss: 0.0017
Step 300/4000 | MLoss: 0.0000 | GLoss: 0.0014
Step 400/4000 | MLoss: 0.0000 | GLoss: 0.0015
Step 500/4000 | MLoss: 0.0000 | GLoss: 0.0008
Step 600/4000 | MLoss: 0.0000 | GLoss: 0.0010
Step 700/4000 | MLoss: 0.0000 | GLoss: 0.0010
Step 800/4000 | M